# Phase 5: Final Results (Minimal)

Goal: aggregate Phase 11 finalized runs into paper-ready comparisons.


# Learnable MSFA Research Track

This notebook is part of the 10-day publication-oriented extension track.

## Run discipline
- Keep one experiment purpose per notebook.
- Save metrics and checkpoints to the configured project root.
- Do not silently change data splits or loss definitions across notebooks.
- When a result becomes final, export both numeric artifacts and a figure/table asset.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/Msa-Osp")
FIG_DIR = PROJECT_ROOT / "phase5_figures"
SUMMARY_SINGLE_PATH = PROJECT_ROOT / "phase5_phase11_single_run_summary.csv"
SUMMARY_MULTI_PATH = PROJECT_ROOT / "phase5_phase11_multi_seed_summary.csv"

HISTORY_FILES = {
    "OSP": PROJECT_ROOT / "phase11_osp_history.csv",
    "Exact OSP": PROJECT_ROOT / "phase11_exact_selector_history.csv",
    "Learnable": PROJECT_ROOT / "phase11_learnable_history.csv",
}

for method, path in HISTORY_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required history for {method}: {path}")

histories = {method: pd.read_csv(path) for method, path in HISTORY_FILES.items()}
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def best_idx_by_psnr(df):
    return int(df["val_psnr"].idxmax())

def best_idx_by_sam(df):
    return int(df["val_sam_deg"].idxmin())

def summarize_single_method(method, df):
    i_psnr = best_idx_by_psnr(df)
    i_sam = best_idx_by_sam(df)
    last = df.iloc[-1]
    return {
        "model": method,
        "epochs_ran": int(len(df)),
        "best_psnr_epoch": int(df.loc[i_psnr, "epoch"]),
        "best_val_psnr": float(df.loc[i_psnr, "val_psnr"]),
        "sam_at_best_psnr": float(df.loc[i_psnr, "val_sam_deg"]),
        "best_sam_epoch": int(df.loc[i_sam, "epoch"]),
        "best_val_sam_deg": float(df.loc[i_sam, "val_sam_deg"]),
        "psnr_at_best_sam": float(df.loc[i_sam, "val_psnr"]),
        "final_epoch": int(last["epoch"]),
        "final_val_psnr": float(last["val_psnr"]),
        "final_val_sam_deg": float(last["val_sam_deg"]),
    }

DETAIL_SINGLE_PATH = PROJECT_ROOT / "phase5_phase11_single_run_detailed.csv"
DETAIL_MULTI_PATH = PROJECT_ROOT / "phase5_phase11_multi_seed_detailed.csv"
PAIRED_PATH = PROJECT_ROOT / "phase5_phase11_multi_seed_paired_improvements.csv"

single_detailed_rows = [summarize_single_method(method, df) for method, df in histories.items()]
single_detailed = pd.DataFrame(single_detailed_rows)

# Add relative improvements for quick paper text.
if "OSP" in single_detailed["model"].values:
    ref_osp = single_detailed.loc[single_detailed["model"] == "OSP"].iloc[0]
    single_detailed["delta_best_psnr_vs_osp"] = single_detailed["best_val_psnr"] - float(ref_osp["best_val_psnr"])
    single_detailed["delta_best_sam_vs_osp"] = single_detailed["best_val_sam_deg"] - float(ref_osp["best_val_sam_deg"])
if "Exact OSP" in single_detailed["model"].values:
    ref_exact = single_detailed.loc[single_detailed["model"] == "Exact OSP"].iloc[0]
    single_detailed["delta_best_psnr_vs_exact"] = single_detailed["best_val_psnr"] - float(ref_exact["best_val_psnr"])
    single_detailed["delta_best_sam_vs_exact"] = single_detailed["best_val_sam_deg"] - float(ref_exact["best_val_sam_deg"])

single_summary = single_detailed[["model", "best_psnr_epoch", "best_val_psnr", "sam_at_best_psnr"]].copy()
single_summary.to_csv(SUMMARY_SINGLE_PATH, index=False)
single_detailed.to_csv(DETAIL_SINGLE_PATH, index=False)

print(f"Saved single-run summary to: {SUMMARY_SINGLE_PATH}")
print(f"Saved single-run detailed table to: {DETAIL_SINGLE_PATH}")
display(single_summary.round(4))
display(single_detailed.round(4))

# Optional multi-seed aggregation: expects names containing seed, e.g., *_seed42.csv.
seed_regex = re.compile(r"seed[_-]?(\\d+)", re.IGNORECASE)
pattern_by_method = {
    "OSP": "phase11_osp_history*.csv",
    "Exact OSP": "phase11_exact_selector_history*.csv",
    "Learnable": "phase11_learnable_history*.csv",
}

multi_rows = []
for method, pattern in pattern_by_method.items():
    files = sorted(PROJECT_ROOT.glob(pattern))
    for f in files:
        m = seed_regex.search(f.stem)
        if m is None and f.name != HISTORY_FILES[method].name:
            continue
        seed_tag = m.group(1) if m else "current"
        df = pd.read_csv(f)
        stats = summarize_single_method(method, df)
        multi_rows.append({"model": method, "seed": seed_tag, **stats})

multi_seed_detailed = pd.DataFrame(multi_rows)

if len(multi_seed_detailed) > 0:
    multi_seed_detailed = multi_seed_detailed.sort_values(["model", "seed"]).reset_index(drop=True)
    multi_seed_detailed.to_csv(DETAIL_MULTI_PATH, index=False)

    agg = (
        multi_seed_detailed.groupby("model", as_index=False)
        .agg(
            n_runs=("seed", "nunique"),
            psnr_mean=("best_val_psnr", "mean"),
            psnr_std=("best_val_psnr", "std"),
            sam_mean=("best_val_sam_deg", "mean"),
            sam_std=("best_val_sam_deg", "std"),
        )
    )
    agg.to_csv(SUMMARY_MULTI_PATH, index=False)

    psnr_pivot = multi_seed_detailed.pivot_table(index="seed", columns="model", values="best_val_psnr", aggfunc="first")
    sam_pivot = multi_seed_detailed.pivot_table(index="seed", columns="model", values="best_val_sam_deg", aggfunc="first")

    paired = pd.DataFrame(index=sorted(set(psnr_pivot.index).union(set(sam_pivot.index))))
    if "Learnable" in psnr_pivot.columns and "OSP" in psnr_pivot.columns:
        paired["learnable_minus_osp_psnr"] = psnr_pivot["Learnable"] - psnr_pivot["OSP"]
    if "Learnable" in psnr_pivot.columns and "Exact OSP" in psnr_pivot.columns:
        paired["learnable_minus_exact_psnr"] = psnr_pivot["Learnable"] - psnr_pivot["Exact OSP"]
    if "Learnable" in sam_pivot.columns and "OSP" in sam_pivot.columns:
        paired["learnable_minus_osp_sam_deg"] = sam_pivot["Learnable"] - sam_pivot["OSP"]
    if "Learnable" in sam_pivot.columns and "Exact OSP" in sam_pivot.columns:
        paired["learnable_minus_exact_sam_deg"] = sam_pivot["Learnable"] - sam_pivot["Exact OSP"]

    paired = paired.dropna(how="all").reset_index().rename(columns={"index": "seed"})
    if len(paired) > 0:
        paired.to_csv(PAIRED_PATH, index=False)
        print(f"Saved paired seed-wise improvements to: {PAIRED_PATH}")
        display(paired.round(4))

    print(f"Saved multi-seed detailed table to: {DETAIL_MULTI_PATH}")
    print(f"Saved multi-seed summary to: {SUMMARY_MULTI_PATH}")
    display(multi_seed_detailed.round(4))
    display(agg.round(4))
else:
    print("No Phase 11 seed-tagged files found for multi-seed aggregation.")


In [ ]:
method_order = ["OSP", "Exact OSP", "Learnable"]

plt.figure(figsize=(10, 4))
for method in method_order:
    if method not in histories:
        continue
    df = histories[method]
    i_psnr = int(df["val_psnr"].idxmax())
    plt.plot(df["epoch"], df["val_psnr"], label=method)
    plt.scatter(float(df.loc[i_psnr, "epoch"]), float(df.loc[i_psnr, "val_psnr"]), s=35)
plt.xlabel("Epoch")
plt.ylabel("Validation PSNR (dB)")
plt.title("Phase 11 Comparison: Validation PSNR")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "phase11_psnr_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 4))
for method in method_order:
    if method not in histories:
        continue
    df = histories[method]
    i_sam = int(df["val_sam_deg"].idxmin())
    plt.plot(df["epoch"], df["val_sam_deg"], label=method)
    plt.scatter(float(df.loc[i_sam, "epoch"]), float(df.loc[i_sam, "val_sam_deg"]), s=35)
plt.xlabel("Epoch")
plt.ylabel("Validation SAM (deg)")
plt.title("Phase 11 Comparison: Validation SAM")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "phase11_sam_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

bar_df = single_detailed.set_index("model").reindex(method_order).dropna(subset=["best_val_psnr", "best_val_sam_deg"])
if len(bar_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(bar_df.index, bar_df["best_val_psnr"])
    axes[0].set_ylabel("Best validation PSNR (dB)")
    axes[0].set_title("Best PSNR by Method")
    axes[0].tick_params(axis="x", rotation=10)

    axes[1].bar(bar_df.index, bar_df["best_val_sam_deg"])
    axes[1].set_ylabel("Best validation SAM (deg)")
    axes[1].set_title("Best SAM by Method")
    axes[1].tick_params(axis="x", rotation=10)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "phase11_best_metric_bars.png", dpi=200, bbox_inches="tight")
    plt.show()

if "agg" in globals() and isinstance(agg, pd.DataFrame) and len(agg) > 0:
    agg_df = agg.set_index("model").reindex(method_order).dropna(subset=["psnr_mean", "sam_mean"])
    if len(agg_df) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].errorbar(
            agg_df.index,
            agg_df["psnr_mean"],
            yerr=agg_df["psnr_std"].fillna(0.0),
            fmt="o",
            capsize=4,
        )
        axes[0].set_ylabel("Best PSNR mean +/- std")
        axes[0].set_title("Multi-seed PSNR")
        axes[0].tick_params(axis="x", rotation=10)

        axes[1].errorbar(
            agg_df.index,
            agg_df["sam_mean"],
            yerr=agg_df["sam_std"].fillna(0.0),
            fmt="o",
            capsize=4,
        )
        axes[1].set_ylabel("Best SAM mean +/- std")
        axes[1].set_title("Multi-seed SAM")
        axes[1].tick_params(axis="x", rotation=10)

        fig.tight_layout()
        fig.savefig(FIG_DIR / "phase11_multiseed_errorbars.png", dpi=200, bbox_inches="tight")
        plt.show()

print(f"Saved figures to: {FIG_DIR}")
